# 第3回: 畳み込みオートエンコーダ (CAE)

本ノートブックでは、**畳み込みオートエンコーダ (Convolutional Autoencoder: CAE)** を学びます。
MNISTデータセットを使い、画像の圧縮・再構成を体験しながら、畳み込みニューラルネットワークの理解を深めましょう。

**目次**
1. モデル構造の理解
2. データセットの準備
3. 学習 (Trainer)
4. MNISTを使った画像圧縮と再構成
5. 番外編: U-NETの実装課題

---
## 1. モデル構造の理解

### 1.1 オートエンコーダの概念

オートエンコーダは、入力データを一度低次元に圧縮（エンコード）し、
その低次元表現から元のデータを復元（デコード）するニューラルネットワークです。

```
入力画像 → [エンコーダ] → ボトルネック（低次元特徴） → [デコーダ] → 再構成画像
 (28x28)                    (feat_dim次元)                    (28x28)
```

- **エンコーダ**: 入力を徐々に小さくし、重要な特徴だけを残す
- **ボトルネック**: 最も圧縮された表現（潜在表現・特徴ベクトル）
- **デコーダ**: 圧縮された表現から元の画像を復元する

学習の目的は「入力 ≈ 出力」となるようにすることです。
これにより、ボトルネック部分にデータの本質的な特徴が学習されます。

### 1.2 Conv2d と ConvTranspose2d

#### Conv2d（畳み込み層）
画像の空間サイズを縮小します。出力サイズの計算式:

$$H_{out} = \left\lfloor \frac{H_{in} + 2 \times \text{padding} - \text{kernel\_size}}{\text{stride}} \right\rfloor + 1$$

例: `nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1)`
- 入力: 28x28 → 出力: $\lfloor(28 + 2 - 3) / 2\rfloor + 1 = 14$ → **14x14**

#### ConvTranspose2d（転置畳み込み層）
画像の空間サイズを拡大します。出力サイズの計算式:

$$H_{out} = (H_{in} - 1) \times \text{stride} - 2 \times \text{padding} + \text{kernel\_size} + \text{output\_padding}$$

例: `nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1)`
- 入力: 4x4 → 出力: $(4-1) \times 2 - 2 + 3 + 1 = 7$ → **7x7** (output_paddingで奇数サイズに調整)

### 1.3 活性化関数

| 関数 | 式 | 特徴 |
|------|-----|------|
| **ReLU** | $\max(0, x)$ | 最も一般的。負の値を0にする |
| **LeakyReLU** | $\max(\alpha x, x)$ | 負の値も小さな勾配を持つ |
| **Tanh** | $\tanh(x)$ | 出力範囲 $[-1, 1]$ |
| **Sigmoid** | $\sigma(x) = \frac{1}{1+e^{-x}}$ | 出力範囲 $[0, 1]$。画像出力層に適する |

今回のCAEでは:
- エンコーダ・デコーダの中間層: **ReLU**
- デコーダの最終層: **Sigmoid**（MNISTの画素値 [0,1] に合わせる）

### 1.4 CAE モデルの実装

MNISTの画像 (28x28, 1チャンネル) に対するCAEを実装します。

エンコーダの空間サイズ変化:
```
28x28 → Conv(s=2) → 14x14 → Conv(s=2) → 7x7 → Conv(s=2) → 4x4 → Flatten → feat_dim
```

デコーダの空間サイズ変化:
```
feat_dim → Linear → 4x4 → ConvT(s=2) → 7x7 → ConvT(s=2) → 14x14 → ConvT(s=2) → 28x28
```

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# 再現性のためにシードを固定
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")

In [ ]:
class CAE(nn.Module):
    """畳み込みオートエンコーダ (Convolutional Autoencoder)"""

    def __init__(self, feat_dim=10):
        super().__init__()
        # エンコーダ: 畳み込み層で空間サイズを縮小
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, 2, 1),   # 28x28 -> 14x14
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, 2, 1),  # 14x14 -> 7x7
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1),  # 7x7 -> 4x4
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, feat_dim),
        )
        # デコーダ: エンコーダの鏡像構造
        self.decoder = nn.Sequential(
            nn.Linear(feat_dim, 64 * 4 * 4),
            nn.ReLU(),
            nn.Unflatten(1, (64, 4, 4)),
            nn.ConvTranspose2d(64, 32, 3, 2, 1, 1),  # 4x4 -> 7x7 (output_padding=1で奇数に)
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, 2, 1, 1),  # 7x7 -> 14x14
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, 2, 1, 1),   # 14x14 -> 28x28
            nn.Sigmoid(),  # 出力を[0,1]に
        )

    def forward(self, x):
        z = self.encoder(x)
        y = self.decoder(z)
        return y, z

### 1.5 モデル構造の確認

`torchinfo.summary` を使ってモデルの構造を確認しましょう。
各層の出力形状とパラメータ数がわかります。

In [ ]:
from torchinfo import summary

model = CAE(feat_dim=10).to(device)
summary(model, input_size=(1, 1, 28, 28), device=str(device))

---
## 2. データセットの準備

### 2.1 MNIST データセット

MNISTは手書き数字 (0-9) のグレースケール画像データセットです。
- 訓練データ: 60,000枚
- テストデータ: 10,000枚
- 画像サイズ: 28x28ピクセル、1チャンネル
- 画素値: 0.0 ~ 1.0（ToTensorで自動正規化）

In [ ]:
# データセットの準備
transform = transforms.ToTensor()  # PIL画像 → Tensor [0,1]

train_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

print(f"訓練データ数: {len(train_dataset)}")
print(f"テストデータ数: {len(test_dataset)}")
print(f"画像の形状: {train_dataset[0][0].shape}")  # (1, 28, 28)

### 2.2 DataLoader

`DataLoader` はデータセットからミニバッチを効率的に取り出すためのユーティリティです。
- `batch_size`: 一度に処理するサンプル数
- `shuffle`: エポックごとにデータの順番をランダムにする
- `num_workers`: データ読み込みの並列プロセス数

In [ ]:
batch_size = 256

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# バッチの取り出し例
images, labels = next(iter(train_loader))
print(f"バッチの形状: {images.shape}")   # (256, 1, 28, 28)
print(f"ラベルの形状: {labels.shape}")   # (256,)
print(f"画素値の範囲: [{images.min():.1f}, {images.max():.1f}]")

### 2.3 データの可視化

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(16):
    ax = axes[i // 8, i % 8]
    ax.imshow(images[i].squeeze(), cmap="gray")
    ax.set_title(f"{labels[i].item()}", fontsize=10)
    ax.axis("off")
plt.suptitle("MNISTサンプル画像", fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. 学習 (Trainer)

### 3.1 損失関数: MSELoss

オートエンコーダの学習では、入力画像と再構成画像の差を最小化します。
**MSE (Mean Squared Error)** を使用:

$$\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} \| x_i - \hat{x}_i \|^2$$

ここで $x_i$ は入力画像、$\hat{x}_i$ は再構成画像です。

### 3.2 学習ループの実装

In [ ]:
def train_cae(model, train_loader, test_loader, epochs=20, lr=1e-3):
    """CAEの学習を行う関数"""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    train_losses = []
    test_losses = []

    for epoch in range(epochs):
        # --- 訓練 ---
        model.train()
        running_loss = 0.0
        for images, _ in train_loader:  # ラベルは使わない（教師なし学習）
            images = images.to(device)

            reconstructed, _ = model(images)
            loss = criterion(reconstructed, images)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        train_losses.append(train_loss)

        # --- 評価 ---
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for images, _ in test_loader:
                images = images.to(device)
                reconstructed, _ = model(images)
                loss = criterion(reconstructed, images)
                test_loss += loss.item() * images.size(0)

        test_loss /= len(test_loader.dataset)
        test_losses.append(test_loss)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1:3d}/{epochs}]  "
                  f"Train Loss: {train_loss:.6f}  Test Loss: {test_loss:.6f}")

    return train_losses, test_losses

### 3.3 損失曲線の可視化

In [ ]:
def plot_losses(train_losses, test_losses):
    """損失曲線を描画する関数"""
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="訓練損失")
    plt.plot(test_losses, label="テスト損失")
    plt.xlabel("エポック")
    plt.ylabel("MSE Loss")
    plt.title("学習曲線")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## 4. MNISTを使った画像圧縮と再構成

### 4.1 学習の実行

feat_dim=10（28x28=784次元 → 10次元に圧縮）で学習します。

In [ ]:
model = CAE(feat_dim=10).to(device)
train_losses, test_losses = train_cae(model, train_loader, test_loader, epochs=20, lr=1e-3)
plot_losses(train_losses, test_losses)

### 4.2 元画像と再構成画像の比較

In [ ]:
def show_reconstruction(model, test_loader, n=8):
    """元画像と再構成画像を並べて表示"""
    model.eval()
    images, labels = next(iter(test_loader))
    images = images[:n].to(device)

    with torch.no_grad():
        reconstructed, _ = model(images)

    fig, axes = plt.subplots(2, n, figsize=(n * 1.5, 3))
    for i in range(n):
        # 元画像
        axes[0, i].imshow(images[i].cpu().squeeze(), cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("元画像", fontsize=10)

        # 再構成画像
        axes[1, i].imshow(reconstructed[i].cpu().squeeze(), cmap="gray")
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("再構成", fontsize=10)

    plt.suptitle("元画像 vs 再構成画像", fontsize=14)
    plt.tight_layout()
    plt.show()

show_reconstruction(model, test_loader)

### 4.3 特徴空間の可視化（feat_dim=2）

特徴次元を2にすると、2次元平面上にデータの分布を可視化できます。
各数字がどのように分布しているか確認しましょう。

In [ ]:
# feat_dim=2 のモデルを学習
model_2d = CAE(feat_dim=2).to(device)
train_losses_2d, test_losses_2d = train_cae(model_2d, train_loader, test_loader, epochs=20, lr=1e-3)
plot_losses(train_losses_2d, test_losses_2d)

In [ ]:
# テストデータ全体の特徴ベクトルを取得
model_2d.eval()
all_z = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        _, z = model_2d(images)
        all_z.append(z.cpu().numpy())
        all_labels.append(labels.numpy())

all_z = np.concatenate(all_z, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

# 散布図
plt.figure(figsize=(8, 6))
scatter = plt.scatter(all_z[:, 0], all_z[:, 1], c=all_labels, cmap="tab10",
                      s=2, alpha=0.5)
plt.colorbar(scatter, label="数字ラベル")
plt.xlabel("特徴次元 1")
plt.ylabel("特徴次元 2")
plt.title("CAE 潜在空間の可視化 (feat_dim=2)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# feat_dim=2 での再構成も確認
show_reconstruction(model_2d, test_loader)

---
## 演習問題

### 演習1: CAEのエンコーダ部分を自分で実装せよ

以下のクラスの `encoder` を完成させてください。
Conv2dの出力サイズを計算しながら、28x28 → 14x14 → 7x7 → 4x4 と縮小するエンコーダを実装しましょう。

**ヒント**: `nn.Sequential` の中に `nn.Conv2d`, `nn.ReLU`, `nn.Flatten`, `nn.Linear` を適切に配置してください。

In [ ]:
class MyCAE(nn.Module):
    def __init__(self, feat_dim=10):
        super().__init__()
        # TODO: エンコーダを実装
        # 28x28 -> 14x14 -> 7x7 -> 4x4 -> feat_dim
        self.encoder = nn.Sequential(
            # ここにコードを書いてください
        )
        # デコーダは提供済み
        self.decoder = nn.Sequential(
            nn.Linear(feat_dim, 64 * 4 * 4),
            nn.ReLU(),
            nn.Unflatten(1, (64, 4, 4)),
            nn.ConvTranspose2d(64, 32, 3, 2, 1, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, 2, 1, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, 2, 1, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        y = self.decoder(z)
        return y, z

# テスト
# my_model = MyCAE(feat_dim=10).to(device)
# x = torch.randn(1, 1, 28, 28).to(device)
# y, z = my_model(x)
# print(f"入力: {x.shape}, 再構成: {y.shape}, 特徴: {z.shape}")

<details>
<summary><b>回答例を表示</b></summary>

```python
class MyCAE(nn.Module):
    def __init__(self, feat_dim=10):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, 2, 1),   # 28x28 -> 14x14
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, 2, 1),  # 14x14 -> 7x7
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1),  # 7x7 -> 4x4
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, feat_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(feat_dim, 64 * 4 * 4),
            nn.ReLU(),
            nn.Unflatten(1, (64, 4, 4)),
            nn.ConvTranspose2d(64, 32, 3, 2, 1, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, 2, 1, 1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, 2, 1, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        y = self.decoder(z)
        return y, z

my_model = MyCAE(feat_dim=10).to(device)
x = torch.randn(1, 1, 28, 28).to(device)
y, z = my_model(x)
print(f"入力: {x.shape}, 再構成: {y.shape}, 特徴: {z.shape}")

```

</details>

### 演習2: 特徴次元(feat_dim)を変えて再構成品質を比較せよ

feat_dim を 2, 10, 50 に変えてそれぞれ学習し、再構成画像を比較してください。
特徴次元が大きいほど再構成品質はどう変わるか観察しましょう。

**ヒント**: 3つのモデルを順番に学習し、`show_reconstruction` で結果を比較してください。

In [ ]:
feat_dims = [2, 10, 50]
models = {}
losses = {}

for fd in feat_dims:
    print(f"\n=== feat_dim = {fd} ===")
    # ここにコードを書いてください
    pass

<details>
<summary><b>回答例を表示</b></summary>

```python
feat_dims = [2, 10, 50]
models = {}
losses = {}

for fd in feat_dims:
    print(f"\n=== feat_dim = {fd} ===")
    m = CAE(feat_dim=fd).to(device)
    tr_loss, te_loss = train_cae(m, train_loader, test_loader, epochs=20, lr=1e-3)
    models[fd] = m
    losses[fd] = (tr_loss, te_loss)

# 損失曲線の比較
plt.figure(figsize=(8, 4))
for fd in feat_dims:
    plt.plot(losses[fd][1], label=f"feat_dim={fd}")
plt.xlabel("エポック")
plt.ylabel("テスト MSE Loss")
plt.title("特徴次元による再構成品質の比較")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 再構成画像の比較
for fd in feat_dims:
    print(f"\nfeat_dim = {fd}:")
    show_reconstruction(models[fd], test_loader)

```

</details>

---
## 5. 番外編: U-NET

### 5.1 U-NETの概念

U-NETは、エンコーダとデコーダの間に**スキップ接続 (Skip Connection)** を持つネットワークです。
通常のオートエンコーダではボトルネックを通過する際に空間情報が失われますが、
スキップ接続により、エンコーダの各レベルの特徴マップをデコーダに直接渡すことで、
細かい空間情報を保持できます。

### 5.2 U-NETの構造

```
入力 (1, 28, 28)
  │
  ├─ enc1: Conv(1→16) + ReLU ──────────────────────┐ skip1 (16, 28, 28)
  │  │                                              │
  │  └─ MaxPool2d(2) → (16, 14, 14)                │
  │     │                                           │
  │     ├─ enc2: Conv(16→32) + ReLU ─────────┐ skip2 (32, 14, 14)
  │     │  │                                  │     │
  │     │  └─ MaxPool2d(2) → (32, 7, 7)      │     │
  │     │     │                               │     │
  │     │     └─ bottleneck: Conv(32→64)      │     │
  │     │        + ReLU → (64, 7, 7)          │     │
  │     │        │                             │     │
  │     │     up2: ConvT(64→32) → (32, 14, 14)│     │
  │     │        │                             │     │
  │     │     cat(up2, skip2) → (64, 14, 14) ─┘     │
  │     │        │                                   │
  │     │     dec2: Conv(64→32) + ReLU               │
  │     │        │                                   │
  │     │     up1: ConvT(32→16) → (16, 28, 28)      │
  │     │        │                                   │
  │     │     cat(up1, skip1) → (32, 28, 28) ────────┘
  │     │        │
  │     │     dec1: Conv(32→16) + ReLU
  │     │        │
  │     │     final: Conv(16→1) → (1, 28, 28)
  │     │        │
  │     │     Sigmoid → 出力 (1, 28, 28)
```

**ポイント**:
- `cat(up, skip)` でチャンネル方向に結合するため、dec層の入力チャンネル数は2倍になる
- スキップ接続により、エンコーダの空間的な詳細情報がデコーダに直接伝わる

### 演習3 (番外編): U-NETを実装し、CAEと再構成品質を比較せよ

以下のヒントを参考に、U-NET型のオートエンコーダを実装してください。
通常のCAEと比較して、再構成品質がどう変わるか確認しましょう。

**注意**: この演習では `forward` メソッド内でスキップ接続を実装する必要があるため、
`nn.Sequential` だけでは実装できません。

**ヒント**: `torch.cat([up, skip], dim=1)` でスキップ接続を実現します。

In [ ]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        # エンコーダ
        self.enc1 = nn.Sequential(nn.Conv2d(1, 16, 3, 1, 1), nn.ReLU())
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(nn.Conv2d(16, 32, 3, 1, 1), nn.ReLU())
        self.pool2 = nn.MaxPool2d(2)
        # ボトルネック
        self.bottleneck = nn.Sequential(nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU())
        # デコーダ
        self.up2 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec2 = nn.Sequential(nn.Conv2d(64, 32, 3, 1, 1), nn.ReLU())
        self.up1 = nn.ConvTranspose2d(32, 16, 2, 2)
        self.dec1 = nn.Sequential(nn.Conv2d(32, 16, 3, 1, 1), nn.ReLU())
        self.final = nn.Sequential(nn.Conv2d(16, 1, 1), nn.Sigmoid())

    def forward(self, x):
        # ここにコードを書いてください
        pass

# テスト
# unet = UNet().to(device)
# x = torch.randn(1, 1, 28, 28).to(device)
# y = unet(x)
# print(f"入力: {x.shape}, 出力: {y.shape}")

<details>
<summary><b>回答例を表示</b></summary>

```python
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        # エンコーダ
        self.enc1 = nn.Sequential(nn.Conv2d(1, 16, 3, 1, 1), nn.ReLU())
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(nn.Conv2d(16, 32, 3, 1, 1), nn.ReLU())
        self.pool2 = nn.MaxPool2d(2)
        # ボトルネック
        self.bottleneck = nn.Sequential(nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU())
        # デコーダ
        self.up2 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec2 = nn.Sequential(nn.Conv2d(64, 32, 3, 1, 1), nn.ReLU())
        self.up1 = nn.ConvTranspose2d(32, 16, 2, 2)
        self.dec1 = nn.Sequential(nn.Conv2d(32, 16, 3, 1, 1), nn.ReLU())
        self.final = nn.Sequential(nn.Conv2d(16, 1, 1), nn.Sigmoid())

    def forward(self, x):
        # エンコーダ
        s1 = self.enc1(x)          # (16, 28, 28)
        p1 = self.pool1(s1)        # (16, 14, 14)
        s2 = self.enc2(p1)         # (32, 14, 14)
        p2 = self.pool2(s2)        # (32, 7, 7)
        # ボトルネック
        b = self.bottleneck(p2)    # (64, 7, 7)
        # デコーダ + スキップ接続
        u2 = self.up2(b)           # (32, 14, 14)
        d2 = self.dec2(torch.cat([u2, s2], dim=1))  # (64, 14, 14) -> (32, 14, 14)
        u1 = self.up1(d2)          # (16, 28, 28)
        d1 = self.dec1(torch.cat([u1, s1], dim=1))  # (32, 28, 28) -> (16, 28, 28)
        out = self.final(d1)       # (1, 28, 28)
        return out

# テスト
unet = UNet().to(device)
x = torch.randn(1, 1, 28, 28).to(device)
y = unet(x)
print(f"入力: {x.shape}, 出力: {y.shape}")

# 学習と比較
optimizer = optim.Adam(unet.parameters(), lr=1e-3)
criterion = nn.MSELoss()

unet_train_losses = []
unet_test_losses = []

for epoch in range(20):
    unet.train()
    running_loss = 0.0
    for images, _ in train_loader:
        images = images.to(device)
        reconstructed = unet(images)
        loss = criterion(reconstructed, images)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    unet_train_losses.append(running_loss / len(train_loader.dataset))

    unet.eval()
    test_loss = 0.0
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            reconstructed = unet(images)
            loss = criterion(reconstructed, images)
            test_loss += loss.item() * images.size(0)
    unet_test_losses.append(test_loss / len(test_loader.dataset))

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/20]  Test Loss: {unet_test_losses[-1]:.6f}")

# U-NET の再構成結果
unet.eval()
images, _ = next(iter(test_loader))
images = images[:8].to(device)
with torch.no_grad():
    recon = unet(images)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(images[i].cpu().squeeze(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon[i].cpu().squeeze(), cmap="gray")
    axes[1, i].axis("off")
axes[0, 0].set_title("元画像")
axes[1, 0].set_title("U-NET再構成")
plt.suptitle("U-NET による再構成結果")
plt.tight_layout()
plt.show()

```

</details>

---
## まとめ

本ノートブックでは以下を学びました:

1. **オートエンコーダの概念**: エンコーダ → ボトルネック → デコーダ の構造
2. **Conv2d / ConvTranspose2d**: 空間サイズの縮小・拡大とサイズ計算
3. **CAEの実装と学習**: MSELossによる再構成学習
4. **特徴空間の可視化**: 低次元に圧縮されたデータの分布
5. **U-NET**: スキップ接続による空間情報の保持

**次回予告**: 第4回では変分オートエンコーダ (VAE) を扱い、生成モデルへの第一歩を踏み出します。